<a href="https://colab.research.google.com/github/ldongheedev/-BDA-LLM-RAG-Program/blob/main/11%EC%A3%BC%EC%B0%A8_%EB%B3%B5%EC%8A%B5%EA%B3%BC%EC%A0%9C_%EC%99%84%EC%84%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 11주차 복습 과제: 임베딩 & 벡터 DB

## 🏨 시나리오: 하루스테이 숙박 FAQ 검색 시스템

```
여러분은 숙박 예약 스타트업 '하루스테이'의 주니어 개발자입니다.
대표의 요청: "고객 문의에 자동으로 관련 FAQ를 찾아주는 검색 시스템을 만들어주세요.
             키워드가 정확히 일치하지 않아도 의미가 비슷하면 찾아야 합니다."
```

실습에서 만든 위니브마켓 FAQ 검색과 **구조가 동일**합니다.

| 실습 매핑 | 위니브마켓(실습) | 하루스테이(과제) |
| --- | --- | --- |
| 문서 데이터 | 쇼핑몰 FAQ | 숙박 FAQ |
| 검색 쿼리 | 배송/교환/포인트 문의 | 체크인/취소/조식 문의 |
| 메타데이터 | 배송, 교환/환불, 포인트 | 예약, 객실, 부대시설, 취소/환불 |

| Part | 내용 | 배점 |
| --- | --- | --- |
| A-1 | 임베딩 변환 직접 확인 | 15점 |
| A-2 | 코사인 유사도 비교 | 15점 |
| A-3 | 벡터 DB 구축 + 유사도 검색 | 20점 |
| A-4 | 메타데이터 필터링 검색 | 20점 |
| B | 직접 구현 (옵션 선택) | 30점 |

> ⚠️ **시작 전 체크**: 아래 `0️⃣` 셀에서 API Key를 입력하세요.

---
## 0️⃣ 환경 설정

In [ ]:
!pip install -q langchain langchain-google-genai langchain-chroma chromadb
print('✅ 설치 완료')

In [ ]:
import os
os.environ['GOOGLE_API_KEY'] = 'google key'

from langchain_google_genai import GoogleGenerativeAIEmbeddings

embedding_model = GoogleGenerativeAIEmbeddings(model='gemini-embedding-001')

print('✅ 임베딩 모델 준비 완료')

In [ ]:
# 하루스테이 FAQ 데이터
faq_data = [
    # 예약 관련
    {'text': '온라인 예약은 체크인 날짜 기준 최소 1일 전까지 가능합니다.', 'category': '예약'},
    {'text': '예약 확인은 마이페이지 > 예약내역에서 확인할 수 있습니다.', 'category': '예약'},
    {'text': '예약 시 신용카드 또는 계좌이체로 결제할 수 있습니다.', 'category': '예약'},
    {'text': '비회원도 예약번호와 전화번호로 예약 조회가 가능합니다.', 'category': '예약'},
    {'text': '동일 날짜에 중복 예약은 불가합니다.', 'category': '예약'},

    # 객실 관련
    {'text': '체크인은 오후 3시, 체크아웃은 오전 11시입니다.', 'category': '객실'},
    {'text': '얼리 체크인은 오후 1시부터 가능하며 추가 요금 2만원이 부과됩니다.', 'category': '객실'},
    {'text': '레이트 체크아웃은 오후 1시까지 가능하며 추가 요금 3만원이 부과됩니다.', 'category': '객실'},
    {'text': '객실 내 무료 와이파이가 제공됩니다.', 'category': '객실'},
    {'text': '모든 객실에 미니바, 커피머신, 금고가 비치되어 있습니다.', 'category': '객실'},
    {'text': '반려동물 동반 투숙은 펫 프렌들리 객실에서만 가능합니다.', 'category': '객실'},
    {'text': '엑스트라 베드는 1대당 3만원이며 사전 요청이 필요합니다.', 'category': '객실'},

    # 부대시설 관련
    {'text': '조식은 1층 레스토랑에서 오전 7시부터 10시까지 운영됩니다.', 'category': '부대시설'},
    {'text': '조식 뷔페 가격은 성인 25,000원, 아동 15,000원입니다.', 'category': '부대시설'},
    {'text': '수영장은 여름 시즌(6~9월) 오전 9시부터 오후 6시까지 운영됩니다.', 'category': '부대시설'},
    {'text': '피트니스 센터는 투숙객에게 무료로 24시간 개방됩니다.', 'category': '부대시설'},
    {'text': '지하 주차장은 투숙객 무료이며, 외부 방문객은 시간당 3,000원입니다.', 'category': '부대시설'},
    {'text': '비즈니스 센터에서 프린트, 복사, 팩스 서비스를 이용할 수 있습니다.', 'category': '부대시설'},
    {'text': '세탁 서비스는 오전 9시까지 맡기면 당일 오후 6시에 수령 가능합니다.', 'category': '부대시설'},

    # 취소/환불 관련
    {'text': '체크인 7일 전까지 취소 시 전액 환불됩니다.', 'category': '취소/환불'},
    {'text': '체크인 3일 전까지 취소 시 숙박비의 70%가 환불됩니다.', 'category': '취소/환불'},
    {'text': '체크인 1일 전 취소 시 숙박비의 50%가 환불됩니다.', 'category': '취소/환불'},
    {'text': '체크인 당일 취소 및 노쇼는 환불이 불가합니다.', 'category': '취소/환불'},
    {'text': '환불은 취소 신청 후 영업일 기준 3~5일 내 처리됩니다.', 'category': '취소/환불'},
    {'text': '태풍, 폭설 등 천재지변 시에는 전액 환불이 적용됩니다.', 'category': '취소/환불'},
]

print(f'✅ FAQ 데이터 준비 완료: {len(faq_data)}개')
for cat in ['예약', '객실', '부대시설', '취소/환불']:
    cnt = sum(1 for d in faq_data if d['category'] == cat)
    print(f'   {cat}: {cnt}개')

---
## 📋 Part A — 빈칸 채우기 (70점)

### A-1. 임베딩 변환 직접 확인 (15점)

**힌트**: 실습 셀 1️⃣~2️⃣를 참고하세요. `embed_query()`는 문장 1개를, `embed_documents()`는 여러 문장을 벡터로 변환합니다.

In [ ]:
# 문장 하나를 벡터로 변환
sentence = '체크인 시간을 변경하고 싶어요'

vector = embedding_model.embed_query(sentence)

print(f'원본 문장: "{sentence}"')
print(f'벡터 차원: {len(vector)}개 숫자')
print(f'앞 5개 값: {[round(v, 4) for v in vector[:5]]}')
print()

# 여러 문장을 한번에 변환
sentences = [
    '체크인 시간을 변경하고 싶어요',
    '입실 시간을 바꿀 수 있나요',
    '조식 뷔페 가격이 얼마예요',
]

vectors = embedding_model.embed_documents(sentences)

print('각 문장의 벡터 앞 3개 값:')
for s, v in zip(sentences, vectors):
    print(f'  "{s}" → {[round(x, 4) for x in v[:3]]}...')

print()
assert len(vector) == 3072, '❌ 벡터 차원이 3072가 아닙니다.'
assert len(vectors) == 3, '❌ 3개 문장의 벡터가 생성되지 않았습니다.'
print('✅ A-1 통과!')

### A-2. 코사인 유사도 비교 (15점)

**힌트**: 실습 셀 4️⃣를 참고하세요. 코사인 유사도가 1에 가까울수록 의미가 비슷합니다.

In [ ]:
import numpy as np

def cosine_similarity(vec1, vec2):
    v1, v2 = np.array(vec1), np.array(vec2)
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

base = '체크인 시간이 몇 시인가요'

compare = [
    ('거의 같은 의미', '입실은 몇 시부터 가능한가요'),
    ('비슷한 주제',     '체크아웃은 언제까지 해야 하나요'),
    ('다른 주제',       '조식 뷔페 운영 시간 알려주세요'),
    ('전혀 다른 주제',  '서울 날씨가 오늘 어떤가요'),
]

base_vec = embedding_model.embed_query(base)

print(f'기준: "{base}"')
print('-' * 55)
for label, sent in compare:
    vec = embedding_model.embed_query(sent)
    sim = cosine_similarity(base_vec, vec)
    bar = '█' * int(sim * 20)
    print(f'  {sim:.4f}  {bar}  ({label})')
    print(f'    "{sent}"')
    print()

assert cosine_similarity(base_vec, embedding_model.embed_query(compare[0][1])) > 0.5, '❌ 유사 문장의 유사도가 너무 낮습니다.'
print('✅ A-2 통과!')

### A-3. 벡터 DB 구축 + 유사도 검색 (20점)

**힌트**: 실습 셀 6️⃣~7️⃣을 참고하세요. `Chroma.from_texts()`로 DB를 만들고 `similarity_search()`로 검색합니다.

In [ ]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_texts(
    texts=[item['text'] for item in faq_data],
    embedding=embedding_model,
    metadatas=[{'category': item['category']} for item in faq_data],
)

print(f'✅ 벡터 DB 구축 완료! 저장 문서: {vectorstore._collection.count()}개')
print()

queries = [
    '몇 시에 들어갈 수 있어요?',
    '예약 취소하면 환불 얼마나 되나요',
    '아침밥 먹을 수 있나요',
    '강아지 데려가도 되나요',
    '주차 가능한가요',
]

print('=== 유사도 검색 결과 ===')
print()
for query in queries:
    print(f'Q: "{query}"')
    results = vectorstore.similarity_search(query, k=2)
    for rank, doc in enumerate(results, 1):
        print(f'  {rank}위: {doc.page_content}')
    print()

assert vectorstore._collection.count() == len(faq_data), '❌ DB에 저장된 문서 수가 맞지 않습니다.'
print('✅ A-3 통과!')

### A-4. 메타데이터 필터링 검색 (20점)

**힌트**: 실습 셀 🔟을 참고하세요. `filter={'category': '카테고리명'}`으로 특정 카테고리만 검색할 수 있습니다.

In [ ]:
query = '환불 받고 싶어요'

print(f'Q: "{query}"')
print()

print('[필터 없이 전체 검색]')
results_all = vectorstore.similarity_search(query, k=3)
for i, doc in enumerate(results_all, 1):
    print(f'  {i}. [{doc.metadata["category"]}] {doc.page_content}')
print()

print('[취소/환불 카테고리만 검색]')
results_filtered = vectorstore.similarity_search(
    query, k=3,
    filter={'category': '취소/환불'},
)
for i, doc in enumerate(results_filtered, 1):
    print(f'  {i}. [{doc.metadata["category"]}] {doc.page_content}')
print()

print('[객실 카테고리만 검색]')
results_room = vectorstore.similarity_search(
    query, k=3,
    filter={'category': '객실'},
)
for i, doc in enumerate(results_room, 1):
    print(f'  {i}. [{doc.metadata["category"]}] {doc.page_content}')

print()
assert all(doc.metadata['category'] == '취소/환불' for doc in results_filtered), '❌ 필터링이 작동하지 않습니다.'
print('✅ A-4 통과!')

---
## 🚀 Part B — 직접 구현 (30점)

아래 두 옵션 중 **하나를 선택**해 직접 구현하세요.

**옵션 1 — 키워드 검색 vs 의미 검색 비교**
하루스테이 FAQ에서 키워드 검색과 의미 검색의 차이를 보여주는 예시를 만드세요.
(힌트: "짐 보관" → 키워드로는 못 찾지만 의미 검색으로는 "세탁 서비스" 등 관련 문서를 찾을 수 있음)

**옵션 2 — 유사도 점수 기반 답변 신뢰도 표시**
`similarity_search_with_score()`를 사용해서 검색 결과에 신뢰도(높음/보통/낮음)를 표시하는 함수를 만드세요.

---

**선택한 옵션:** `옵션 1`

In [ ]:
# Part B 구현 — 옵션 1: 키워드 검색 vs 의미 검색 비교

def keyword_search(query, documents, k=3):
    query_chars = set(query.replace(' ', ''))
    results = []
    for doc in documents:
        doc_chars = set(doc.replace(' ', ''))
        overlap = len(query_chars & doc_chars)
        if overlap > 0:
            results.append((overlap, doc))
    results.sort(reverse=True)
    return [doc for _, doc in results[:k]]

faq_texts = [item['text'] for item in faq_data]

test_queries = [
    '짐 맡길 수 있나요',
    '아이랑 같이 가는데 침대 추가 가능한가요',
    '운동할 데 있어요?',
    '비가 많이 와서 못 갈 것 같아요',
]

print('=== 키워드 검색 vs 의미 검색 비교 ===')
print()

for query in test_queries:
    print(f'Q: "{query}"')
    print()

    kw_results = keyword_search(query, faq_texts, k=2)
    print('  [키워드 검색]')
    if kw_results:
        for i, doc in enumerate(kw_results, 1):
            print(f'    {i}. {doc}')
    else:
        print('    (검색 결과 없음)')

    sem_results = vectorstore.similarity_search(query, k=2)
    print('  [의미 검색]')
    for i, doc in enumerate(sem_results, 1):
        print(f'    {i}. {doc.page_content}')

    print()
    print('-' * 60)
    print()

In [ ]:
# Part B 결과 분석

print('=== 비교 분석 ===')
print()
print('키워드 검색의 한계:')
print('  - "짐 맡길 수 있나요" → "짐", "맡기다"라는 단어가 FAQ에 없어서 못 찾음')
print('  - "운동할 데 있어요?" → "운동"이라는 단어가 없어서 피트니스 센터를 못 찾음')
print()
print('의미 검색의 장점:')
print('  - "짐 맡기다" → "세탁 서비스", "비즈니스 센터" 등 관련 서비스를 찾아냄')
print('  - "운동할 데" → "피트니스 센터" 문서를 의미적으로 연결')
print('  - "비가 많이 와서 못 갈 것 같아요" → "천재지변 시 전액 환불" 문서를 찾아냄')
print()
print('→ 의미 검색은 단어가 달라도 맥락이 비슷하면 찾을 수 있습니다!')

---
## 📝 회고

In [ ]:
review = """
[과제를 하면서 느낀 점]
- 실습과 다르게 헷갈렸던 부분: embed_query()와 embed_documents()의 차이. 하나는 문장 1개, 하나는 리스트를 받는다는 점.
- 가장 이해가 잘 된 개념: 키워드 검색과 의미 검색의 차이. "짐 맡기다"로 검색했을 때 키워드로는 못 찾지만 의미 검색으로는 관련 서비스를 찾는 것이 인상적이었습니다.
- Part B에서 선택한 옵션과 이유: 옵션 1을 선택했습니다. 키워드 검색과 의미 검색을 직접 비교하면 임베딩의 장점이 확실히 드러나기 때문입니다.
"""
print(review)

---
## ✅ 제출 전 체크리스트

- [ ] API Key를 `YOUR_API_KEY`로 다시 바꿨나요?
- [ ] A-1 ~ A-4 모두 `✅ 통과!` 메시지를 확인했나요?
- [ ] Part B 옵션 중 하나를 선택해 구현하고 테스트 출력을 남겼나요?
- [ ] 회고를 작성했나요?